# 27. 학습: 제외어 목록 방식(규칙 필터)과 누출 문제

> 이 노트북은 **학습용**입니다. 완성 코드 대신 개념 설명, 우리 프로젝트에서 쓴 이유, 연습문제(TODO)를 담았습니다. 답은 직접 채워 보세요.

## 1. 한 줄 요약
검색 카드가 **언제** 답해도 되는지를 낱말 규칙으로 정하는 방식입니다. 필수어·질문어·제외어 세 목록으로 판단합니다.

## 2. 우리 프로젝트의 규칙 (run.py 카드)
| 항목 | 뜻 | 예 (홍역 Q03 카드) |
|---|---|---|
| 필수어 | 질문에 **반드시** 있어야 함 | `홍역`, `검사` |
| 질문어 | **두 개 이상** 있어야 함 | `환자`, `등교`, `아닌`, `확인` |
| 제외어 | 하나라도 있으면 **안 씀** | `수족구`, `풍진`, `결핵`, `격리` … |
| 검색 후보 | 블록이 검색 상위 후보에 있어야 함 | 전문 검색 + 소제목 검색 |

네 조건을 모두 통과해야 카드가 답합니다.

## 3. 수업에서 배운 내용인가? (확인 결과)
- **네, 방향은 수업에 있습니다.** `2_graphrag/2_그래프분석과검색/38-5_세-방법-비교와-라우터-벡터RAG-GraphRAG-Text2Cypher_배포용.ipynb`의 **3. 라우터**가 같은 생각입니다.
  - 수업의 규칙 라우터: "몇 개·몇 건·모두" 같은 낱말이 있으면 Text2Cypher, 없으면 GraphRAG.
  - 수업도 규칙의 한계를 가르칩니다. **연습문제 3 "규칙이 놓치는 질문"**: 낱말 하나를 바꾸면 다른 질문까지 흔들린다는 점입니다.
  - 수업의 **실습 B**는 같은 일을 LLM 라우터로 합니다(질문을 `llm.invoke`로 분류).
- **그대로 쓴 것은 아닙니다.** 수업은 "질문 유형을 고르는 라우터"이고, 우리는 "카드가 답해도 되는지 거르는 필터"입니다. 같은 원리(낱말 규칙)를 다른 자리에 적용했습니다.
- 수업 밖(42~45일차)의 내용은 이 방식과 관계없고, 이번에는 참고만 했습니다.

## 4. 왜 이 방식을 골랐나 (채택 이유)
1. **설명하기 쉽다.** 발표 때 "이 낱말이 있으면 이 카드, 이 낱말이 있으면 막는다"로 바로 보여 줄 수 있습니다.
2. **빠르고 외부 호출이 없다.** LLM 라우터는 질문마다 호출 비용과 지연이 생깁니다. 11/6~11/9 시연 일정에서는 부담이 큽니다.
3. **결과가 항상 같다.** 같은 질문이면 같은 답입니다. 평가 재현성에 유리합니다.
4. **근거 중심의 보수적 방식이다.** 애매하면 답하지 않습니다(INSUFFICIENT). 근거 없는 답을 막는 목적에 맞습니다.

## 5. 한계 (반드시 알고 있어야 함)
- **목록 밖 낱말을 막지 못함.** 예: 목록에 없는 질병명이 질문에 섞이면, 홍역 카드가 붙을 수 있습니다. 우리는 풍진·결핵을 추가했지만, 다른 질병이 나오면 또 추가해야 합니다.
- **동의어·띄어쓰기에 약함.** "MMR2차"처럼 붙여 쓰면 낱말이 맞지 않을 수 있습니다.
- **같은 낱말의 다른 뜻을 구분하지 못함.** "환자"가 "환자 아님"인지 "환자 등교 금지"인지 낱말만으로는 구분이 약합니다.
- **목록 관리 비용.** 카드가 늘수록 목록을 같이 손봐야 합니다.

## 6. 대안 비교
| 방식 | 장점 | 단점 | 수업 근거 |
|---|---|---|---|
| 낱말 규칙 (지금) | 빠르고 설명 쉬움 | 목록 밖 누출, 의미 구분 약함 | 38-5 규칙 라우터 |
| LLM 라우터 | 의도를 더 잘 구분 | 호출 비용, 결과 흔들림 | 38-5 실습 B |
| 질병 사전·메타데이터 필터 | 질병이 명확히 정해져 누출이 적음 | 사전·속성을 준비해야 함 | 팀 사전(질병 목록) 사용 가능 여부 확인 필요 |

## 7. 연습문제 (TODO — 직접 해 보세요)
아래 질문은 모두 **홍역 카드가 붙으면 안 되는** 질문입니다. 현재 규칙으로 실행해 보고, 결과를 적어 보세요.


In [1]:
# 연습 준비: 카드 규칙을 한 줄씩 보는 함수 (학습용 복사본, 실제 run.py 와 같은 규칙)
def card_passes(card, question, retrieved=True):
    """카드가 이 질문에 답해도 되는지 규칙으로 판단한다.
    retrieved: 검색 후보에 카드 블록이 들어 있었는지 (여기서는 True 로 가정)"""
    if not retrieved:
        return False                                        # 검색 후보에 없으면 끝
    if not all(term in question for term in card.get("필수어", [])):
        return False                                        # 필수어가 모두 있어야 한다
    if sum(term in question for term in card["질문어"]) < 2:
        return False                                        # 질문어는 두 개 이상
    if any(term in question for term in card.get("제외어", [])):
        return False                                        # 제외어가 하나라도 있으면 막는다
    return True

# 실제 홍역 Q03 카드 규칙 (run.py ITEM2B_CARD 와 같은 값)
Q03_CARD = {
    "필수어": ["홍역", "검사"],
    "질문어": ["환자", "등교", "아닌", "확인"],
    "제외어": ["전파", "잠복기", "증상", "감염경로", "부작용", "효과", "간격", "접촉", "예방접종",
              "격리", "수족구", "성홍열", "백일해", "수두", "코로나", "독감", "인플루엔자", "유행성이하선염", "풍진", "결핵"],
}
print(card_passes(Q03_CARD, "홍역 검사 결과 환자가 아닌 것으로 확인되면 학생은 언제 등교하나요?"))  # 기대: True


True


### 연습 1. 누출 찾기 (TODO)
아래 질문 중 **Q03 카드가 붙는(True) 질문**을 찾으세요. 왜 붙는지 한 문장으로 적으세요.

- A. "홍역 검사는 음성이지만 풍진 환자는 언제 등교하나요?"
- B. "홍역 검사 결과 환자가 아닌데 HIV 감염자는 언제 등교하나요?"
- C. "홍역 검사 결과 환자가 아니면 전염병 환자는 언제 등교하나요?"

> 힌트: 제외어 목록에 있는 낱말과 없는 낱말을 나눠 보세요. 목록에 없는 질병 이름이 나오면 어떻게 될까요?


In [4]:
# TODO: 세 질문을 card_passes 로 돌려 보고 결과를 적으세요.
# questions = [...]
# for q in questions: print(q, card_passes(Q03_CARD, q))

questions = ["홍역 검사는 음성이지만 풍진 환자는 언제 등교하나요"]
for q in questions: print(q, card_passes(Q03_CARD, q))


홍역 검사는 음성이지만 풍진 환자는 언제 등교하나요 False


In [5]:
questions = ["홍역 검사 결과 환자가 아닌데 HIV 감염자는 언제 등교하나요?"]
for q in questions: print(q, card_passes(Q03_CARD, q))

홍역 검사 결과 환자가 아닌데 HIV 감염자는 언제 등교하나요? True


In [6]:
questions = ["홍역 검사 결과 환자가 아니면 전염병 환자는 언제 등교하나요?"]
for q in questions: print(q, card_passes(Q03_CARD, q))

홍역 검사 결과 환자가 아니면 전염병 환자는 언제 등교하나요? True


### 연습 2. 바꿔 보기 (TODO)
연습 1에서 누출된 질문이 있다면, **제외어에 낱말을 추가**하면 막힙니다. 그런데 추가하면 정상적인 질문도 막힐 수 있습니다.

- 질문: "홍역 검사 결과 음성이면 격리 없이 등교할 수 있나요?" 는 정상 질문인가요?
- 제외어에 "격리"가 있으면 이 질문은 어떻게 되나요? 이것이 **제외어 방식의 부작용**입니다.

> 힌트: 제외어를 넓히면 안전해지지만, 정상 질문을 놓칩니다. 이 균형이 방식의 핵심 어려움입니다.


내용 이해했고, 제외어가 늘어나면 질문에 대한 답변이 좋아질 수 있으나, 반대로 정말 답해야하는 질문도 걸러낼 수 있어 한계가 확실하다.

### 연습 3. 더 나은 방법 설계 (TODO, 생각해 보기)
질병 이름을 목록으로 막는 대신, **질문에서 질병을 먼저 찾고** 카드의 질병과 다르면 막는 방법을 생각해 보세요.

1. 질병 목록(예: `MEASLES`, `HFMD`, `SCARLET_FEVER`)은 어디서 가져올 수 있을까요? (팀의 사전 파일 확인)
 호흡기감염병 관리지침에서 확인 (질병 사전 기반 방법)

2. 질문에서 질병을 찾는 규칙을 어떻게 만들까요? 
명명된 단어, 증상, 관련 지침

3. 카드에 질병 속성을 붙이면(예: `disease = MEASLES`), 검색 결과 중 **같은 질병 카드만** 남기는 것이 가능할까요? 
새로운 질병이름이 추가되면 가능할 것 같다

4. 이 방식의 약점은 무엇일까요? (질병 이름이 질문에 없을 때)
실제 이름이 질병명이 없거나, 제외한 단어가 많아지면 제대로 된 답변이 나오지 않을 수 있다

> 이 연습은 설계만 합니다. 실제 적용은 팀과 상의한 뒤에 합니다(사전·속성 변경은 팀 합의 대상).


## 8. 정리 (노션에 옮기기 좋게)
- 개념: 낱말 규칙 필터(필수어·질문어·제외어)로 카드가 답해도 되는지 정한다.
- 수업 근거: 38-5 규칙 라우터와 "규칙이 놓치는 질문", 실습 B의 LLM 라우터.
- 채택 이유: 설명·속도·재현성, 애매하면 답하지 않는 보수적 설계.
- 한계: 목록 밖 낱말 누출, 동의어·띄어쓰기, 의미 구분 약함, 목록 관리 비용.
- 다음 방향: 질병 사전·속성 기반 필터 또는 LLM 라우터와 비교 실험(팀 합의 후).
